# Identify state priorities for household WASH deprivation in Nigeria

This notebook compares two 2024 state estimates on the **same eligible households**. The direct estimate uses observed WASH outcomes. The model estimate averages household probabilities from six held-out zone folds. The WASH outcome is deprivation in at least two of the three domains among households with all three domains known. The 2013 round remains in the project and is outside this 2024 validation.

**Before running**

1. Place `Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz` next to this notebook, or inside `climate_stack_outputs` in the same folder. Leave it compressed. The model uses household predictors, so the code reads only the required columns.
2. Set the priority threshold in the first code cell. The example uses **50%** for a practice run. Replace it with a programme threshold chosen before inspecting the rankings if one is available. All classifications are conditional on that choice.
3. Open the notebook in JupyterLab and run the cells in order. If an import is missing, run `%pip install numpy pandas scipy scikit-learn matplotlib pillow python-docx` in a separate code cell, restart the kernel and run the notebook again.
4. The map cell reuses the earlier ADM1 boundary. If the automatic download fails, use [this manual GeoJSON download](https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1_simplified.geojson) and place the file beside the notebook with the name `geoBoundaries-NGA-ADM1_simplified.geojson`.

Outputs go to `wash_state_priority_outputs`. The report contains editable tables and both figures. The notebook also saves the figures as separate 600 dpi JPEGs and verifies their resolution.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler
from scipy.optimize import brentq
from scipy.special import expit, logit

# Use a programme threshold selected before reading the results, if available.
PRIORITY_THRESHOLD_PCT = 50.0
THRESHOLD_NOTE = "Illustrative threshold for a practice run"
# Replace the note above when you enter a programme threshold.
if not 0 < PRIORITY_THRESHOLD_PCT < 100:
    raise ValueError("Enter a threshold between 0 and 100.")

ROOT = Path.cwd()
INPUT = ROOT / "Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz"
if not INPUT.is_file():
    INPUT = ROOT / "climate_stack_outputs" / INPUT.name
if not INPUT.is_file():
    raise FileNotFoundError("Put the compressed climate-linked household file beside the notebook.")
OUTPUT = ROOT / "wash_state_priority_outputs"
OUTPUT.mkdir(exist_ok=True)
KEY = ["survey_round","hv001","hv002"]
TARGET = "wash_dep_2plus"
DEMOGRAPHIC = ["rural","head_female","head_age_years_capped","usual_household_size","nonwash_asset_count"]
RAW = ["head_age_years_capped","usual_household_size","nonwash_asset_count"]
columns = [*KEY,"state","zone","residence","head_sex","hh_weight","psu_key","stratum_hv023_key","wash_domains_known",TARGET,*RAW]
d = pd.read_csv(INPUT,usecols=columns,low_memory=False)
if d.groupby("survey_round").size().to_dict() != {2013:38522,2018:40427,2024:40047}:
    raise ValueError("Unexpected survey counts. Select the combined household output from the stack notebook.")
if d[KEY].isna().any().any() or d.duplicated(KEY).any():
    raise ValueError("Missing or duplicate household keys.")
if not np.isfinite(d.hh_weight).all() or not d.hh_weight.gt(0).all():
    raise ValueError("Household weights must be positive and finite.")
if not d[TARGET].dropna().isin([0,1]).all() or d.loc[d.survey_round.eq(2013),TARGET].notna().any():
    raise ValueError("The file does not use the verified main 2018–2024 target.")
main_rounds = d.survey_round.isin([2018,2024])
if not d.loc[main_rounds,TARGET].notna().eq(d.loc[main_rounds,"wash_domains_known"].eq(3)).all():
    raise ValueError("The target does not match the complete-case eligibility rule.")
if d[["state","zone"]].isna().any().any() or d.groupby("state").zone.nunique().ne(1).any():
    raise ValueError("Each state needs a consistent zone label.")
for column in RAW:
    d[column] = pd.to_numeric(d[column],errors="raise")
    if not np.isfinite(d[column].dropna()).all():
        raise ValueError(f"Non-finite predictor values in {column}.")
if not d.residence.dropna().isin(["Rural","Urban"]).all() or not d.head_sex.dropna().isin(["Female","Male"]).all():
    raise ValueError("Unexpected residence or head-sex labels.")
d["rural"] = d.residence.map({"Rural":1.,"Urban":0.})
d["head_female"] = d.head_sex.map({"Female":1.,"Male":0.})
analysis = d.loc[main_rounds & d[TARGET].notna()].copy()
analysis[TARGET] = analysis[TARGET].astype(int)
if analysis.groupby("survey_round").size().to_dict() != {2018:36821,2024:36412}:
    raise ValueError("Unexpected complete-case totals. Review the target and household input.")

## 1. Recreate the held-out 2024 probabilities

In each fold, the notebook fits the same weighted logistic model on 2018 households from the other five zones. It fits the intercept adjustment on 2024 outcomes from those same five zones and then scores the sixth zone. This makes the estimate for each state independent of outcomes from its zone during fitting and adjustment. The national checks should reproduce the previous recalibration run before the notebook proceeds.

In [ ]:
ZONES = ["North Central","North East","North West","South East","South South","South West"]
train2018 = analysis.loc[analysis.survey_round.eq(2018)]
test2024 = analysis.loc[analysis.survey_round.eq(2024)].copy()
if set(train2018.zone) != set(ZONES) or set(test2024.zone) != set(ZONES):
    raise ValueError("The six zones must appear in both modelling rounds.")

def fit_base(train):
    median = train[DEMOGRAPHIC].median()
    if median.isna().any():
        raise ValueError("A training predictor is entirely missing.")
    x = train[DEMOGRAPHIC].fillna(median)
    scaler = StandardScaler().fit(x)
    model = LogisticRegression(C=1,max_iter=1000)
    model.fit(scaler.transform(x),train[TARGET],sample_weight=train.hh_weight/train.hh_weight.mean())
    return median,scaler,model

def score_base(frame, fitted):
    median,scaler,model = fitted
    x = scaler.transform(frame[DEMOGRAPHIC].fillna(median))
    return model.predict_proba(x)[:,1]


test2024["predicted_original"] = np.nan
test2024["predicted"] = np.nan
fold_rows = []
for zone in ZONES:
    train = train2018.loc[train2018.zone.ne(zone)]
    calibration = test2024.loc[test2024.zone.ne(zone)]
    test = test2024.loc[test2024.zone.eq(zone)]
    if (set(train.state) | set(calibration.state)) & set(test.state):
        raise ValueError(f"State leakage in the {zone} fold.")
    if calibration[TARGET].nunique() != 2 or test2024.loc[test.index, "predicted"].notna().any():
        raise ValueError(f"Invalid calibration outcomes or duplicated test records in {zone}.")
    model = fit_base(train)
    z_cal = logit(np.clip(score_base(calibration, model), 1e-8, 1-1e-8))
    y_cal = calibration[TARGET].to_numpy()
    w_cal = calibration.hh_weight.to_numpy()
    w_cal = w_cal / w_cal.sum()
    offset = brentq(lambda a: np.dot(w_cal, expit(z_cal + a) - y_cal), -60, 60)
    raw = score_base(test, model)
    test2024.loc[test.index, "predicted_original"] = raw
    test2024.loc[test.index, "predicted"] = expit(logit(np.clip(raw, 1e-8, 1-1e-8)) + offset)
    fold_rows.append({"test_zone": zone, "train_2018_n": len(train),
                      "calibration_2024_n": len(calibration), "test_2024_n": len(test),
                      "calibration_intercept": offset})

folds = pd.DataFrame(fold_rows)
if not np.isfinite(test2024[["predicted_original", "predicted"]]).all().all():
    raise ValueError("Missing or invalid held-out predictions.")
y = test2024[TARGET]
w = test2024.hh_weight
observed = np.average(y, weights=w)
qc = pd.DataFrame([
    {"method": label, "test_n": len(y), "observed_pct": 100 * observed,
     "predicted_pct": 100 * np.average(test2024[col], weights=w),
     "brier": brier_score_loss(y, test2024[col], sample_weight=w)}
    for label, col in [("Original", "predicted_original"), ("Intercept only", "predicted")]
])
if not np.allclose(qc.brier, [0.2191944114, 0.2063618652], atol=1e-5, rtol=0):
    raise ValueError("The national Brier scores differ from the verified recalibration run. Check the input and package versions.")
print("National checks against the earlier notebook")
display(qc.round(4))

## 2. Estimate deprivation by state and classify priority

The direct 95% intervals use 999 stratified bootstrap replicates of the original 2024 primary sampling units (PSUs). Each replicate resamples within the survey strata, then computes a weighted ratio within each state among households with known outcomes. These are pointwise intervals for the direct estimates. The priority categories use those intervals and do not represent simultaneous inference across all 37 areas.

A state is **Above threshold** if its lower interval bound is above the cutoff. It is **Below threshold** if its upper bound is below the cutoff. All other states are **Uncertain**. The model flags states by their predicted point estimate. Its validation uncertainty depends on the outcomes used for model fitting and calibration, so the maps do not assign model-based confidence categories.

The table also shows weighted missing outcomes and extreme-case bounds obtained by setting all unknown outcomes to 0 or to 1. These are a sensitivity check for missing outcomes, separate from sampling uncertainty. The full DHS sample contributes to the design and missingness checks.

In [ ]:
PSU, STRATUM = "psu_key","stratum_hv023_key"
full2024 = d.loc[d.survey_round.eq(2024)].copy()
if full2024[[PSU, STRATUM]].isna().any().any():
    raise ValueError("2024 design identifiers contain missing values.")
design = full2024[[PSU, STRATUM]].drop_duplicates().sort_values([STRATUM,PSU]).reset_index(drop=True)
if design[PSU].duplicated().any():
    raise ValueError("A 2024 PSU belongs to more than one stratum.")
if design.groupby(STRATUM).size().lt(2).any():
    raise ValueError("This bootstrap requires at least two PSUs per 2024 stratum.")

B = 999
rng = np.random.default_rng(20241112)
rep = np.zeros((B,len(design)))
for _,members in design.groupby(STRATUM,sort=False):
    positions = members.index.to_numpy()
    n = len(positions)
    rep[:,positions] = rng.multinomial(n-1,np.repeat(1/n,n),size=B)*n/(n-1)


if len(design) != 1380 or not np.allclose(rep.sum(axis=1), len(design)):
    raise ValueError("The 2024 design or bootstrap totals differ from the verified input.")
if full2024.groupby(PSU).state.nunique().ne(1).any():
    raise ValueError("A PSU appears in multiple states.")
if full2024.state.nunique() != 37 or "FCT" not in set(full2024.state):
    raise ValueError("Expected 36 states and the FCT.")

psu_lookup = pd.Series(design.index, index=design[PSU])
rows = []
for state, all_households in full2024.groupby("state"):
    eligible = test2024.loc[test2024.state.eq(state)]
    if eligible.empty:
        raise ValueError(f"No eligible test households in {state}.")
    weights = eligible.hh_weight.to_numpy()
    outcomes = eligible[TARGET].to_numpy()
    probability = eligible.predicted.to_numpy()
    indices = eligible[PSU].map(psu_lookup).to_numpy(dtype=int)
    denominator = np.bincount(indices, weights=weights, minlength=len(design))
    numerator = np.bincount(indices, weights=weights * outcomes, minlength=len(design))
    boot_denominator = rep @ denominator
    if (boot_denominator <= 0).any():
        raise ValueError(f"One or more design replicates have no eligible households in {state}.")
    lo, hi = np.quantile(100 * (rep @ numerator) / boot_denominator, [.025, .975])
    observed_pct = 100 * np.average(outcomes, weights=weights)
    predicted_pct = 100 * np.average(probability, weights=weights)
    all_weight = all_households.hh_weight.sum()
    unknown_weight = all_households.loc[all_households[TARGET].isna(), "hh_weight"].sum()
    missing_lower = 100 * numerator.sum() / all_weight
    missing_upper = 100 * (numerator.sum() + unknown_weight) / all_weight
    rows.append({
        "state": state, "zone": all_households.zone.iloc[0],
        "all_n": len(all_households), "eligible_n": len(eligible),
        "missing_n": len(all_households) - len(eligible),
        "psus_all": all_households[PSU].nunique(), "psus_eligible": eligible[PSU].nunique(),
        "weighted_missing_pct": 100 * unknown_weight / all_weight,
        "observed_pct": observed_pct, "observed_ci95_low_pct": lo, "observed_ci95_high_pct": hi,
        "model_pct": predicted_pct, "model_bias_pp": predicted_pct - observed_pct,
        "missing_lower_pct": missing_lower, "missing_upper_pct": missing_upper,
    })

states = pd.DataFrame(rows).sort_values(["observed_pct", "state"], ascending=[False, True]).reset_index(drop=True)
threshold = PRIORITY_THRESHOLD_PCT
states["observed_rank"] = states.observed_pct.rank(method="min", ascending=False).astype(int)
states["model_rank"] = states.model_pct.rank(method="min", ascending=False).astype(int)
states["survey_class"] = np.select(
    [states.observed_ci95_low_pct.gt(threshold), states.observed_ci95_high_pct.lt(threshold)],
    ["Above threshold", "Below threshold"], default="Uncertain")
states["survey_point_flag"] = states.observed_pct.ge(threshold)
states["model_flag"] = states.model_pct.ge(threshold)
states["point_flags_agree"] = states.survey_point_flag.eq(states.model_flag)
states["missing_bounds_cross_threshold"] = (
    states.missing_lower_pct.le(threshold) & states.missing_upper_pct.ge(threshold))
states["review"] = np.select(
    [states.survey_class.eq("Above threshold") & ~states.model_flag,
     states.survey_class.eq("Below threshold") & states.model_flag,
     states.survey_class.eq("Uncertain") & states.model_flag,
     states.survey_class.eq("Uncertain") & ~states.model_flag],
    ["Model misses survey priority", "Model flags survey-below area",
     "Model flags survey-uncertain area", "Survey uncertain"], default="Agreement")

if states.eligible_n.sum() != len(test2024) or states.all_n.sum() != len(full2024):
    raise ValueError("State counts do not reconcile with national counts.")
if not (states.missing_lower_pct.le(states.observed_pct + 1e-9) &
        states.missing_upper_pct.ge(states.observed_pct - 1e-9)).all():
    raise ValueError("Missing-outcome bounds do not contain the direct estimate.")

comparison = (
    pd.crosstab(states.survey_class, states.model_flag)
    .reindex(index=["Above threshold", "Uncertain", "Below threshold"], columns=[True, False], fill_value=0)
    .rename(columns={True: "Model flags", False: "Model does not flag"})
)
comparison.index.name = "Survey class"
summary = pd.DataFrame([{
    "threshold_pct": threshold, "threshold_note": THRESHOLD_NOTE, "areas": len(states),
    "survey_above": states.survey_class.eq("Above threshold").sum(),
    "survey_uncertain": states.survey_class.eq("Uncertain").sum(),
    "survey_below": states.survey_class.eq("Below threshold").sum(),
    "model_flags": states.model_flag.sum(),
    "survey_point_and_model_agree_n": states.point_flags_agree.sum(),
    "spearman_state_rank": states.observed_pct.corr(states.model_pct, method="spearman"),
    "state_mean_absolute_error_pp": states.model_bias_pp.abs().mean(),
    "missing_bounds_cross_threshold_n": states.missing_bounds_cross_threshold.sum(),
}])
tables = {
    "State_Priority_Estimates_2024.csv": states,
    "State_Priority_Shortlist_2024.csv": states.loc[states.survey_class.eq("Above threshold")],
    "State_Priority_Comparison_2024.csv": comparison.reset_index(),
    "State_Priority_Summary_2024.csv": summary,
    "State_Priority_Folds_2024.csv": folds,
    "State_Priority_National_QC_2024.csv": qc,
}
for filename, frame in tables.items():
    frame.to_csv(OUTPUT / filename, index=False)
display(comparison)
display(states[["state", "observed_pct", "observed_ci95_low_pct", "observed_ci95_high_pct",
                "model_pct", "model_bias_pp", "weighted_missing_pct", "survey_class", "review"]].round(2))

## 3. Map the results

The first figure shows observed and predicted prevalence on the same colour scale. The second shows survey priority categories next to model flags. Hatching on the model flag map marks flagged areas whose survey interval crosses the threshold. Use the CSV for state names and exact values. If your boundary download fails, the CSVs from the previous cell remain available.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import json
import os
import shutil
import subprocess
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.patches import Patch, PathPatch
from matplotlib.path import Path as PlotPath
import numpy as np
import pandas as pd
from PIL import Image

BOUNDARY = ROOT / "geoBoundaries-NGA-ADM1_simplified.geojson"
URL = ("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/"
       "releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1_simplified.geojson")
CREDIT = "GRID3 Nigeria ADM1 via geoBoundaries gbOpen, 2022 boundaries, CC BY 4.0"
if not BOUNDARY.is_file():
    try:
        with urlopen(URL, timeout=45) as response:
            raw = response.read()
        json.loads(raw)
    except (OSError, ValueError) as error:
        raise RuntimeError(f"Use the manual link above and save the boundary as {BOUNDARY.name}.") from error
    BOUNDARY.write_bytes(raw)
raw = BOUNDARY.read_bytes()
geo = json.loads(raw)
if geo.get("type") != "FeatureCollection":
    raise ValueError("Expected a GeoJSON FeatureCollection.")
features, boundary_rows = {}, []
for feature in geo["features"]:
    label = feature["properties"]["shapeName"]
    state = "FCT" if label == "Abuja Federal Capital Territory" else label
    if state in features:
        raise ValueError(f"Duplicate boundary name: {state}")
    features[state] = feature["geometry"]
    boundary_rows.append({"state": state, "boundary_label": label})
state_names = set(states.state)
if set(features) != state_names:
    raise ValueError(f"Unmatched DHS names: {sorted(state_names - features.keys())}. "
                     f"Unmatched boundary names: {sorted(features.keys() - state_names)}.")

OUTPUT.mkdir(parents=True, exist_ok=True)
pd.DataFrame(boundary_rows).sort_values("state").to_csv(OUTPUT / "State_Priority_Boundary_Match.csv", index=False)
(OUTPUT / "Boundary_Source.txt").write_text(
    f"{CREDIT}\n{URL}\nSHA-256 {hashlib.sha256(raw).hexdigest()}\n", encoding="utf-8"
)

dpi = 600

def save_figure(figure, filename):
    path = OUTPUT / filename
    figure.savefig(path, format="jpeg", dpi=dpi, facecolor="white",
                   pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(figure)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (dpi, dpi):
            raise ValueError(f"Could not verify JPEG resolution for {path.name}.")
    return path



def polygon_path(geometry):
    kind = geometry["type"]
    if kind == "Polygon":
        polygons = [geometry["coordinates"]]
    elif kind == "MultiPolygon":
        polygons = geometry["coordinates"]
    else:
        raise ValueError(f"Unexpected geometry type: {kind}")
    paths = []
    for polygon in polygons:
        for i, ring in enumerate(polygon):
            points = np.asarray(ring, dtype=float)[:, :2]
            if len(points) < 4 or not np.isfinite(points).all():
                raise ValueError("Malformed boundary ring.")
            if not np.array_equal(points[0], points[-1]):
                points = np.vstack([points, points[0]])
            x, y = points.T
            signed_area = np.sum(x[:-1] * y[1:] - y[:-1] * x[1:])
            # Opposite winding keeps interior holes open.
            if (signed_area > 0) != (i == 0):
                points = points[::-1]
            codes = np.full(len(points), PlotPath.LINETO, dtype=np.uint8)
            codes[0], codes[-1] = PlotPath.MOVETO, PlotPath.CLOSEPOLY
            paths.append(PlotPath(points, codes))
    return PlotPath.make_compound_path(*paths)

paths = {state: polygon_path(geometry) for state, geometry in features.items()}
coordinates = np.vstack([path.vertices for path in paths.values()])
west, south = coordinates.min(axis=0)
east, north = coordinates.max(axis=0)
if not (0 < west < east < 20 and 0 < south < north < 20):
    raise ValueError("Unexpected boundary extent. Use Nigeria ADM1 longitude and latitude.")
aspect = 1 / np.cos(np.deg2rad((south + north) / 2))


def paint(ax, colors, title, hatched=()):
    for state, path in paths.items():
        ax.add_patch(PathPatch(path, facecolor=colors.loc[state], edgecolor="#364152",
                               linewidth=.42, hatch="////" if state in hatched else None))
    ax.set(xlim=(west-.15, east+.15), ylim=(south-.15, north+.15), aspect=aspect)
    ax.set_axis_off()
    ax.set_title(title, fontsize=11, fontweight="bold", pad=4)

by_state = states.set_index("state")
fig, axes = plt.subplots(1, 2, figsize=(8.3, 4.45))
fig.subplots_adjust(left=.035, right=.965, top=.84, bottom=.23, wspace=.08)
fig.suptitle("Household WASH deprivation by state", fontsize=15, fontweight="bold", y=.965)
fig.text(.5, .89, "Nigeria DHS 2024  |  Deprived in at least two WASH domains", ha="center", fontsize=9)
cmap, norm = plt.get_cmap("YlOrRd"), Normalize(0, 100)
paint(axes[0], by_state.observed_pct.map(lambda x: cmap(norm(x))), "Survey-weighted observed")
paint(axes[1], by_state.model_pct.map(lambda x: cmap(norm(x))), "Held-out model estimate")
bar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                   cax=fig.add_axes([.25, .155, .50, .035]), orientation="horizontal",
                   ticks=[0, 25, 50, 75, 100])
bar.set_label("Households with WASH deprivation (%)", fontsize=9)
fig.text(.5, .023, CREDIT, fontsize=7, ha="center", color="#475569")
prevalence_figure = save_figure(fig, "Figure_1_State_Prevalence_600dpi.jpg")

palette = {"Above threshold": "#B44727", "Uncertain": "#F7CF70", "Below threshold": "#DCE5EB"}
flag_colors = {True: palette["Above threshold"], False: palette["Below threshold"]}
fig, axes = plt.subplots(1, 2, figsize=(8.3, 4.75))
fig.subplots_adjust(left=.035, right=.965, top=.80, bottom=.24, wspace=.08)
fig.suptitle("State priority classification", fontsize=15, fontweight="bold", y=.965)
fig.text(.5, .875, f"Nigeria DHS 2024  |  Threshold {threshold:g}%  |  {THRESHOLD_NOTE}",
         ha="center", fontsize=9, wrap=True)
paint(axes[0], by_state.survey_class.map(palette), "Survey classification with 95% intervals")
hatched = by_state.index[by_state.survey_class.eq("Uncertain") & by_state.model_flag]
paint(axes[1], by_state.model_flag.map(flag_colors), "Model classification from point estimates")
for x, handles in [
    (.255, [Patch(facecolor=color, edgecolor="#364152", label=label) for label, color in palette.items()]),
    (.745, [Patch(facecolor=flag_colors[True], edgecolor="#364152", label="Model flags"),
            Patch(facecolor=flag_colors[False], edgecolor="#364152", label="Model does not flag"),
            Patch(facecolor=flag_colors[True], edgecolor="#364152", hatch="////",
                  label="Model flags, survey uncertain")]),
]:
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(x, .235), frameon=False, fontsize=8)
fig.text(.5, .023, CREDIT, fontsize=7, ha="center", color="#475569")
priority_figure = save_figure(fig, "Figure_2_State_Priority_600dpi.jpg")
print("Saved and verified both 600 dpi JPEGs.")

## 4. Export the Word report and use the shortlist

Start with `State_Priority_Shortlist_2024.csv`. Use `State_Priority_Estimates_2024.csv` for the complete table, including differences, missing outcomes, and confidence intervals. Review the states flagged by the model alone and the survey-priority states missed by the model. Use the survey estimate and its uncertainty for the initial 2024 shortlist. These results cover states and the FCT. An LGA or a later survey would require its own validation.

The report always saves as `.docx` and also exports a true Word 97–2003 `.doc` when LibreOffice Writer is available. If LibreOffice is unavailable, open the generated `.docx` in desktop Word and choose **Save As → Word 97–2003 Document (.doc)**.

In [ ]:
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

figure_width = 7
doc = Document()
doc.core_properties.title = "State priorities for WASH deprivation in Nigeria"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.75)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 16), ("bottom", 16), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")




doc.add_paragraph("State priorities for WASH deprivation in Nigeria", style="Title")
doc.add_paragraph("2024 Nigeria Demographic and Health Survey")
doc.add_paragraph(
    f"The {threshold:g}% cutoff is labelled ‘{THRESHOLD_NOTE}’. "
    f"The survey benchmark places {int(summary.survey_above.iloc[0])} of 37 areas above the cutoff, "
    f"with {int(summary.survey_uncertain.iloc[0])} uncertain. "
    f"The model flags {int(summary.model_flags.iloc[0])} areas. "
    "The estimates compare the same eligible households using the same survey weights."
)
add_table("Table 1. Survey classification compared with model flags",
          ["Survey class", "Model flags", "Model does not flag"],
          [[label, *map(int, row)] for label, row in comparison.iterrows()],
          [2.4, 2.2, 2.4], {0})
high = states.head(10)
rows = [[r.state, f"{r.observed_pct:.1f} ({r.observed_ci95_low_pct:.1f}–{r.observed_ci95_high_pct:.1f})",
         f"{r.model_pct:.1f}", f"{r.model_bias_pp:+.1f}", f"{r.weighted_missing_pct:.1f}", r.survey_class]
        for r in high.itertuples(index=False)]
add_table("Table 2. Ten areas with the highest direct prevalence estimates",
          ["State", "Survey % (95% CI)", "Model %", "Bias (pp)", "Missing %", "Survey class"],
          rows, [1.20, 1.60, .78, .75, .85, 1.82], {0, 5})
doc.add_paragraph(
    f"Across the 37 areas, the rank correlation is {summary.spearman_state_rank.iloc[0]:.2f} and the "
    f"mean absolute state error is {summary.state_mean_absolute_error_pp.iloc[0]:.1f} percentage points. "
    "Positive bias means the model overestimates deprivation. Missing percentages use the survey weights. "
    "The full results and survey shortlist are in the accompanying CSVs."
)
doc.add_heading("Where observed and predicted prevalence differ", level=1).paragraph_format.page_break_before = True
add_figure(prevalence_figure, "Figure 1. Survey and held-out model estimates on a common scale.",
           "Two Nigeria state maps showing observed and predicted WASH deprivation prevalence.")
add_figure(priority_figure, "Figure 2. Survey priority categories and model flags.",
           "Two Nigeria state maps showing the survey priority classes and the model threshold flags.")
doc.add_paragraph(
    "Direct intervals use a stratified PSU bootstrap. They reflect sampling uncertainty for the eligible "
    "sample. The extreme-case missing outcome bounds are provided separately in the estimates CSV. "
    "The model comparison uses held-out 2024 zones. Future-round performance is untested."
)
def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom and Path(custom).is_file():
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)


docx_path = OUTPUT / "Nigeria_DHS_WASH_State_Priority_Report.docx"
doc.save(docx_path)

def export_legacy_word(source, soffice):
    with TemporaryDirectory() as scratch:
        temp = Path(scratch).resolve()
        result = subprocess.run(
            [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
             "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source.resolve())],
            capture_output=True, text=True, timeout=180,
        )
        converted = temp / source.with_suffix(".doc").name
        if result.returncode or not converted.is_file():
            raise RuntimeError(result.stderr or result.stdout or "Converter returned no file.")
        with converted.open("rb") as stream:
            if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
                raise ValueError("The converter did not create a binary Word 97–2003 document.")
        shutil.copyfile(converted, source.with_suffix(".doc"))

soffice = find_soffice()
if soffice:
    try:
        export_legacy_word(docx_path, soffice)
        print(f"Saved {docx_path.with_suffix('.doc').resolve()}")
    except (RuntimeError, ValueError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Word .doc conversion failed: {error}")
        print("Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
else:
    print("LibreOffice was unavailable. Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")


print(f"Saved the report, CSVs and two 600 dpi JPEGs in {OUTPUT.resolve()}.")